[Guide Index](README.md)


# 03a SPARQL Entailment

`query()` takes an `entailment=` argument that applies an *entailment regime* to a single query, without first running `infer()` on the graph and without changing the graph. It accepts `ENTAILMENT.RDF`, `ENTAILMENT.RDFS`, `ENTAILMENT['OWL-RDF-Based']` and `ENTAILMENT['OWL-Direct']`. Every entailment regime - what each one means, and a worked example of `infer()` - is explained once, in full, in the [Graph Inferencing guide](02b-graphs-inferencing.ipynb). This guide covers what is specific to the query-time form: how each regime is applied, how to keep the entailed triples between queries (`infer=`) and a copy of the data between calls (a session), how to detect an inconsistency via `ASK`, and what is rejected.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
import pyoxigraph as ox
from starlayer.graph import ENTAILMENT
from starlayer.graph.oxgraph import StarLayerGraph

EX = "http://example.org/"
RDF_TYPE = ox.NamedNode("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

## 1. Entailment in one query

Pass the regime to `query()`; the graph itself is never modified.

- `ENTAILMENT.RDF` and `ENTAILMENT.RDFS`: the query is rewritten so that Oxigraph answers it as if the entailed triples were present. Nothing is copied.
- `ENTAILMENT['OWL-RDF-Based']` (OWL 2 RL, through `owlrl`) and `ENTAILMENT['OWL-Direct']` (OWL 2 DL, through HermiT or `rustdl`): the store is copied, the entailed triples are added to the copy, and the query runs on the copy. The data in the copy is always the current data.

In [2]:
g = StarLayerGraph(prefixes={"ex": EX})
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

QUERY = "SELECT ?x WHERE { ?x a ex:Employee }"

print("plain (no entailment):  ", [x.x.value for x in g.query(QUERY)])
print("entailment=RDFS:        ", [x.x.value for x in g.query(QUERY, entailment=ENTAILMENT.RDFS)])
print("entailment=OWL RL:      ", [x.x.value for x in g.query(QUERY, entailment=ENTAILMENT['OWL-RDF-Based'])])
print("entailment=RDF:         ", [x.x.value for x in g.query(QUERY, entailment=ENTAILMENT.RDF)], "(RDF entailment has no subclass rule)")
print("graph unchanged:", len(g), "triples")

plain (no entailment):   []
entailment=RDFS:         ['http://example.org/alice']
entailment=OWL RL:       ['http://example.org/alice']
entailment=RDF:          [] (RDF entailment has no subclass rule)
graph unchanged: 2 triples


## 2. The two-step form

`g.infer(regime)` computes the same closure as a graph you can keep. By default (`mode="full"`) it returns a **new** `StarLayerGraph` holding the original triples plus the entailed ones; the graph you called it on is not changed. Query that closure instead of the original. (`mode="delta"` returns only the entailed triples and `mode="in-place"` adds them to the original.) Use this form when you want to look at, serialize or reuse the entailed triples themselves; use `entailment=` when you only want answers.

The RDF regime is computed natively on Oxigraph terms; RDFS and OWL RL run through `owlrl` on a temporary rdflib copy, and the result is converted back.

In [3]:
closure = g.infer(ENTAILMENT.RDFS)
print("query on the RDFS closure: ", [x.x.value for x in closure.query(QUERY)])
print("original graph unchanged:", len(g), "triples; closure:", len(closure), "triples")

query on the RDFS closure:  ['http://example.org/alice']
original graph unchanged: 2 triples; closure: 17 triples


## 3. Keeping the entailed triples: `infer=`

Computing the OWL entailments can take a while (an OWL-Direct query on a graph of a thousand individuals spent several seconds in the reasoner). The graph therefore keeps the entailed triples of `ENTAILMENT['OWL-RDF-Based']` and, for each engine, of `ENTAILMENT['OWL-Direct']`, and `infer=` says when they are computed again. The data is always read live; only the entailed triples are reused.

- `infer="changed"` (the default): compute them again if the graph was written to since they were computed.
- `infer="cached"`: reuse them whenever there are any, even after the data changed.
- `infer="always"`: compute them on every call.

A write counts when it goes through StarLayer (`add`, `parse`, `update`, `as_store()`, ...). A write on a raw `pyoxigraph.Store` reference you kept from before is not seen; call `starlayer.graph.oxgraph.union.touch()` after it. Any write to any StarLayer graph counts, so `"changed"` can compute again when it need not; it never reuses stale triples. `infer=` has no effect on `RDF` and `RDFS`, which keep nothing.

In [4]:
# alice is a manager, but she is also a teamlead, because manager and teamlead are the same
g_owl = StarLayerGraph(prefixes={"ex": EX})
g_owl.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
""", format="turtle")

QUERY_OWL = "SELECT ?x WHERE { ?x a ex:TeamLead }"
OWL_RL = ENTAILMENT['OWL-RDF-Based']

def names(rows):
    return sorted(r.x.value.rsplit("/", 1)[1] for r in rows)

print("before:", names(g_owl.query(QUERY_OWL, entailment=OWL_RL)))

g_owl.add((ox.NamedNode(EX + "carol"), RDF_TYPE, ox.NamedNode(EX + "Manager")))
print('infer="cached"  (keeps the old entailed triples - carol is a manager, not yet a teamlead):',
      names(g_owl.query(QUERY_OWL, entailment=OWL_RL, infer="cached")))
print('infer="changed" (the write was seen, so they are computed again):                        ',
      names(g_owl.query(QUERY_OWL, entailment=OWL_RL, infer="changed")))
print('infer="always"  (computed again whatever happened):                                      ',
      names(g_owl.query(QUERY_OWL, entailment=OWL_RL, infer="always")))

before: ['alice']
infer="cached"  (keeps the old entailed triples - carol is a manager, not yet a teamlead): ['alice']
infer="changed" (the write was seen, so they are computed again):                         ['alice', 'carol']
infer="always"  (computed again whatever happened):                                       ['alice', 'carol']


## 4. A query session: one copy, then end it

Each `query(entailment=ENTAILMENT['OWL-RDF-Based'])` or `['OWL-Direct']` call copies the whole store into a throwaway store and adds the entailed triples to it (on a store of a million triples that copy took about six seconds, every call, even when the entailed triples were kept). When you ask several questions of the same graph, open a **session** instead:

- `g.entailment_session(regime)` makes the copy once. The reasoner runs only if the graph does not already keep the entailed triples (`infer=` as in section 3).
- `session.query(...)` runs on that copy, as often as you like, and costs the query only.
- `session.close()` ends the session: it empties the copy and frees its memory. A `with` block does this for you, also when an error is raised. **Always end a session when you are done with it**, because it holds a full copy of the graph's data. The graph itself, and the entailed triples it keeps, are not affected.

A session is a snapshot of the data as it was when it was opened. After a write, `session.stale` is `True` and the session still answers from its snapshot; `session.refresh()` makes a new copy of the current data (the entailed triples are computed again only if the graph changed, or as `refresh("always")` / `refresh("cached")` say). Read a result before you close or refresh the session. Sessions exist for `ENTAILMENT['OWL-RDF-Based']` and `ENTAILMENT['OWL-Direct']`; `RDF` and `RDFS` rewrite the query and keep nothing to hold. A `StarQuery` without its own `entailment=` can be run on a session with `query.run(session)`.

In [5]:
with g_owl.entailment_session(OWL_RL) as session:
    print("open:", not session.closed, "| stale:", session.stale)
    print("first query: ", names(session.query(QUERY_OWL)))
    print("second query:", names(session.query(QUERY_OWL)))

    g_owl.add((ox.NamedNode(EX + "dave"), RDF_TYPE, ox.NamedNode(EX + "Manager")))
    print("after a write - stale:", session.stale, "| still its snapshot:", names(session.query(QUERY_OWL)))

    session.refresh()
    print("after refresh - stale:", session.stale, "|", names(session.query(QUERY_OWL)))

print("after the with block - closed:", session.closed)
try:
    session.query(QUERY_OWL)
except RuntimeError as error:
    print("RuntimeError:", error)

open: True | stale: False
first query:  ['alice', 'carol']
second query: ['alice', 'carol']
after a write - stale: True | still its snapshot: ['alice', 'carol']
after refresh - stale: False | ['alice', 'carol', 'dave']
after the with block - closed: True
RuntimeError: this entailment session is closed; open another with graph.entailment_session(...)


## 5. Detecting inconsistencies via `ASK`

`ENTAILMENT['OWL-RDF-Based']` inherits `owlrl`'s consistency checks (see the Inferencing guide's [section 4.5](02b-graphs-inferencing.ipynb#4.5-Detecting-inconsistencies) for what an inconsistency means and how it's flagged - an `err:error` triple, not an exception). A query still returns a result when the graph is inconsistent, but the flag is among the entailed triples, so a simple `ASK` detects it:

In [6]:
ASK_INCONSISTENT = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
ASK { ?e a err:ErrorMessage }
"""

# Dog and Cat are declared disjoint, but fido is asserted to be both
g_bad = StarLayerGraph(prefixes={"ex": EX})
g_bad.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
''', format="turtle")

print("g_bad (genuine inconsistency):  ", g_bad.query(ASK_INCONSISTENT, entailment=OWL_RL).askAnswer)
print("g_bad without entailment=:      ", g_bad.query(ASK_INCONSISTENT).askAnswer)
print("g_owl (consistent):             ", g_owl.query(ASK_INCONSISTENT, entailment=OWL_RL).askAnswer)

# a normal query still returns a result - nothing is retracted, the inconsistency isn't surfaced here
rows = g_bad.query("SELECT ?c WHERE { ex:fido a ?c }", entailment=OWL_RL)
print("fido's classes (still returned despite the inconsistency):", sorted(r.c.value.rsplit("/", 1)[1] for r in rows))

g_bad (genuine inconsistency):   True
g_bad without entailment=:       False
g_owl (consistent):              False
fido's classes (still returned despite the inconsistency): ['Cat', 'Dog']


## 6. What is rejected

`entailment=` takes only the four regimes above; any other value (such as `ENTAILMENT.D`, which is a recognized regime IRI but not implemented) raises `NotImplementedError` rather than returning an answer that ignores the regime. `engine=` and `timeout=` choose the reasoner (`"hermit"` or `"rustdl"`) and limit its time, and apply to `ENTAILMENT['OWL-Direct']` only; with anything else they raise `ValueError`. The OWL-Direct reasoners are described in the [DL reasoning guide](appendix-b2-dl-reasoning-examples.ipynb).

In [7]:
for label, call in [
    ("entailment=D:", lambda: g.query(QUERY, entailment=ENTAILMENT.D)),
    ("engine= without OWL-Direct:", lambda: g.query(QUERY, entailment=ENTAILMENT.RDFS, engine="rustdl")),
]:
    try:
        call()
    except (NotImplementedError, ValueError) as error:
        print(f"{label:30s}", type(error).__name__ + ":", str(error)[:110])

entailment=D:                  NotImplementedError: entailment must be one of ENTAILMENT.RDF, RDFS, ['OWL-RDF-Based'], ['OWL-Direct'], got <NamedNode value=http:/
engine= without OWL-Direct:    ValueError: engine='rustdl' is only meaningful for entailment=ENTAILMENT['OWL-Direct'] (got entailment=<NamedNode value=ht
